# K-Means and Hierarchical Clustering

## Customer Segmentation (K-Means)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

data = pd.DataFrame({
    'Income': [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30],
    'Spend': [39, 81, 6, 77, 40, 76, 35, 94, 3, 72, 14, 99, 15, 50, 28, 59]
})

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
data['Cluster'] = kmeans.fit_predict(data)

print(data)

plt.scatter(data['Income'], data['Spend'], c=data['Cluster'], cmap='rainbow')
plt.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1], color='black', marker='X', s=200)
plt.xlabel('Income')
plt.ylabel('Spend')
plt.title('K-Means Clustering')
plt.show()

## Customer Segmentation (Hierarchical Clustering)

In [ ]:
import numpy as np
import pandas as pd
from scipy.cluster.hierarchy import dendrogram, linkage
import matplotlib.pyplot as plt

data = pd.DataFrame({
    'Income': [15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30],
    'Spend': [39, 81, 6, 77, 40, 76, 35, 94, 3, 72, 14, 99, 15, 50, 28, 59]
})

linked = linkage(data, method='ward')

plt.figure(figsize=(10, 7))
dendrogram(linked, orientation='top', distance_sort='descending', show_leaf_counts=True)
plt.title('Dendrogram for Hierarchical Clustering')
plt.show()

## Clustering Comparison on Iris Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import (
    KMeans,
    AgglomerativeClustering,
    DBSCAN,
    MeanShift,
    Birch,
    SpectralClustering,
    AffinityPropagation
)
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score,
    adjusted_rand_score,
    normalized_mutual_info_score
)

iris = load_iris()
X = iris.data
y_true = iris.target
feature_names = iris.feature_names
target_names = iris.target_names

df = pd.DataFrame(X, columns=feature_names)
df["Actual_Class"] = y_true

print("First 5 rows:")
print(df.head())
print("\nDataset shape:", X.shape)
print("Target classes:", target_names)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
print("\nPCA explained variance ratio:")
print(pca.explained_variance_ratio_)

models = {
    "K-Means": KMeans(n_clusters=3, random_state=42, n_init=10),
    "Agglomerative": AgglomerativeClustering(n_clusters=3),
    "DBSCAN": DBSCAN(eps=0.8, min_samples=5),
    "Mean Shift": MeanShift(),
    "BIRCH": Birch(n_clusters=3),
    "Gaussian Mixture": GaussianMixture(n_components=3, random_state=42),
    "Spectral": SpectralClustering(n_clusters=3, affinity="nearest_neighbors", random_state=42),
    "Affinity Propagation": AffinityPropagation(random_state=42)
}

def evaluate_clustering(X, labels, y_true):
    unique_labels = set(labels)
    cluster_count = len(unique_labels - {-1})
    result = {
        "Clusters": cluster_count,
        "Silhouette": np.nan,
        "Calinski_Harabasz": np.nan,
        "Davies_Bouldin": np.nan,
        "ARI": adjusted_rand_score(y_true, labels),
        "NMI": normalized_mutual_info_score(y_true, labels)
    }
    if cluster_count >= 2:
        result["Silhouette"] = silhouette_score(X, labels)
        result["Calinski_Harabasz"] = calinski_harabasz_score(X, labels)
        result["Davies_Bouldin"] = davies_bouldin_score(X, labels)
    return result

results = []
cluster_outputs = {}
for name, model in models.items():
    print("\nTraining:", name)
    labels = model.fit_predict(X_scaled)
    cluster_outputs[name] = labels
    metrics = evaluate_clustering(X_scaled, labels, y_true)
    metrics["Algorithm"] = name
    results.append(metrics)

results_df = pd.DataFrame(results)
results_df = results_df[[
    "Algorithm",
    "Clusters",
    "Silhouette",
    "Calinski_Harabasz",
    "Davies_Bouldin",
    "ARI",
    "NMI"
]]
print("\nClustering Validation Results:")
print(results_df)

plt.figure(figsize=(7, 5))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=y_true)
plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")
plt.title("Actual Iris Classes")
plt.grid(True)
plt.show()

for name, labels in cluster_outputs.items():
    plt.figure(figsize=(7, 5))
    plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels)
    plt.xlabel("PCA Component 1")
    plt.ylabel("PCA Component 2")
    plt.title(f"{name} Clustering Result")
    plt.grid(True)
    plt.show()

plt.figure(figsize=(10, 5))
plt.bar(results_df["Algorithm"], results_df["Silhouette"])
plt.xlabel("Clustering Algorithm")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score Comparison")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(results_df["Algorithm"], results_df["Calinski_Harabasz"])
plt.xlabel("Clustering Algorithm")
plt.ylabel("Calinski-Harabasz Score")
plt.title("Calinski-Harabasz Score Comparison")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(results_df["Algorithm"], results_df["Davies_Bouldin"])
plt.xlabel("Clustering Algorithm")
plt.ylabel("Davies-Bouldin Score")
plt.title("Davies-Bouldin Score Comparison")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.show()

plt.figure(figsize=(10, 5))
plt.bar(results_df["Algorithm"], results_df["ARI"])
plt.xlabel("Clustering Algorithm")
plt.ylabel("Adjusted Rand Index")
plt.title("ARI Comparison with Actual Iris Labels")
plt.xticks(rotation=45)
plt.grid(axis="y")
plt.show()

## Clustering Parameter Tuning

### Elbow Method for K-Means

In [ ]:
inertia_values = []
k_values = range(1, 11)
for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertia_values.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_values, inertia_values, marker="o")
plt.xlabel("Number of Clusters K")
plt.ylabel("Inertia")
plt.title("Elbow Method for Choosing K")
plt.grid(True)
plt.show()

### Silhouette Method for K-Means

In [ ]:
silhouette_values = []
k_values = range(2, 11)
for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_values.append(score)

plt.figure(figsize=(8, 5))
plt.plot(k_values, silhouette_values, marker="o")
plt.xlabel("Number of Clusters K")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Method for Choosing K")
plt.grid(True)
plt.show()